# 📊 Analisis Tren Belanja Konsumen 2026
## Studi Kasus: Bisnis & Ekonomi — Analisis Perilaku Pelanggan

---
**Mata Kuliah:** Analisis Data dan Visualisasi Lanjutan  
**Program Studi:** S2 Sains Data, Universitas Hasanuddin  
**Dosen:** Prof. Sri Astuti Thamrin, Ph.D  
**Tahun:** 2026  

---
## 🎯 Rumusan Masalah

Di era transformasi digital, perilaku belanja konsumen bergeser secara masif. Retailer dan e-commerce perlu memahami:
1. **Bagaimana** distribusi belanja online vs toko berdasarkan demografi?
2. **Faktor apa** yang mendorong preferensi saluran belanja tertentu?
3. **Segmen pelanggan** mana yang paling bernilai untuk bisnis?
4. **Apa rekomendasi strategis** berbasis data untuk pemangku kepentingan bisnis?

> *"Insight yang baik harus mengubah keputusan."* — Filosofi Visualisasi Data


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Pengaturan tampilan
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print("✅ Library berhasil dimuat!")


✅ Library berhasil dimuat!


---
## 1. Memuat & Mempersiapkan Data

Data diambil dari dataset *Consumer Shopping Trends 2026* yang mencakup 11.789 responden dengan 25 variabel perilaku belanja.
Semua nama variabel ditransformasi ke Bahasa Indonesia agar mudah dipahami.


In [2]:
# Memuat dataset
df_raw = pd.read_csv("Consumer_Shopping_Trends_2026 (6).csv")

# Terjemahkan nama kolom ke Bahasa Indonesia
kolom_baru = {
    "age":                         "usia",
    "monthly_income":              "pendapatan_bulanan",
    "daily_internet_hours":        "jam_internet_harian",
    "smartphone_usage_years":      "lama_pakai_smartphone",
    "social_media_hours":          "jam_media_sosial",
    "online_payment_trust_score":  "kepercayaan_pembayaran_online",
    "tech_savvy_score":            "skor_melek_teknologi",
    "monthly_online_orders":       "pesanan_online_bulanan",
    "monthly_store_visits":        "kunjungan_toko_bulanan",
    "avg_online_spend":            "rata_belanja_online",
    "avg_store_spend":             "rata_belanja_toko",
    "discount_sensitivity":        "sensitivitas_diskon",
    "return_frequency":            "frekuensi_retur",
    "avg_delivery_days":           "rata_hari_pengiriman",
    "delivery_fee_sensitivity":    "sensitivitas_ongkir",
    "free_return_importance":      "pentingnya_retur_gratis",
    "product_availability_online": "ketersediaan_produk_online",
    "impulse_buying_score":        "skor_pembelian_impulsif",
    "need_touch_feel_score":       "skor_perlu_sentuh_produk",
    "brand_loyalty_score":         "skor_loyalitas_merek",
    "environmental_awareness":     "kesadaran_lingkungan",
    "time_pressure_level":         "tingkat_tekanan_waktu",
    "gender":                      "jenis_kelamin",
    "city_tier":                   "tingkat_kota",
    "shopping_preference":         "preferensi_belanja",
}

df = df_raw.rename(columns=kolom_baru)

# Terjemahkan nilai kategorikal
df["jenis_kelamin"]      = df["jenis_kelamin"].map({"Male": "Pria", "Female": "Wanita", "Other": "Lainnya"})
df["tingkat_kota"]       = df["tingkat_kota"].map({"Tier 1": "Kota Tier 1", "Tier 2": "Kota Tier 2", "Tier 3": "Kota Tier 3"})
df["preferensi_belanja"] = df["preferensi_belanja"].map({"Online": "Online", "Store": "Toko", "Hybrid": "Hybrid"})

# Variabel derivasi
df["total_belanja"]    = df["rata_belanja_online"] + df["rata_belanja_toko"]
df["rasio_online"]     = df["rata_belanja_online"] / df["total_belanja"]

bins_usia  = [17, 25, 35, 45, 55, 65, 80]
label_usia = ["18-25", "26-35", "36-45", "46-55", "56-65", "66-79"]
df["kelompok_usia"] = pd.cut(df["usia"], bins=bins_usia, labels=label_usia, ordered=True)

print(f"Ukuran dataset: {df.shape[0]:,} baris × {df.shape[1]} kolom")
print(f"\nKolom yang tersedia:")
for col in df.columns:
    print(f"  • {col} ({df[col].dtype})")


Ukuran dataset: 11,789 baris × 28 kolom

Kolom yang tersedia:
  • usia (int64)
  • pendapatan_bulanan (int64)
  • jam_internet_harian (float64)
  • lama_pakai_smartphone (int64)
  • jam_media_sosial (float64)
  • kepercayaan_pembayaran_online (int64)
  • skor_melek_teknologi (int64)
  • pesanan_online_bulanan (int64)
  • kunjungan_toko_bulanan (int64)
  • rata_belanja_online (int64)
  • rata_belanja_toko (int64)
  • sensitivitas_diskon (int64)
  • frekuensi_retur (int64)
  • rata_hari_pengiriman (int64)
  • sensitivitas_ongkir (int64)
  • pentingnya_retur_gratis (int64)
  • ketersediaan_produk_online (int64)
  • skor_pembelian_impulsif (int64)
  • skor_perlu_sentuh_produk (int64)
  • skor_loyalitas_merek (int64)
  • kesadaran_lingkungan (int64)
  • tingkat_tekanan_waktu (int64)
  • jenis_kelamin (object)
  • tingkat_kota (object)
  • preferensi_belanja (object)
  • total_belanja (int64)
  • rasio_online (float64)
  • kelompok_usia (category)


---
## 2. Eksplorasi Data Awal (EDA)

In [3]:
# Ringkasan statistik (variabel numerik utama)
cols_tampil = ["usia", "pendapatan_bulanan", "total_belanja",
               "pesanan_online_bulanan", "kunjungan_toko_bulanan",
               "sensitivitas_diskon", "skor_loyalitas_merek", "jam_internet_harian"]

ringkasan = df[cols_tampil].describe().T[["mean", "50%", "std", "min", "max"]]
ringkasan.columns = ["Rata-rata", "Median", "Std Dev", "Min", "Max"]
ringkasan.index.name = "Variabel"
print("Statistik Deskriptif — Variabel Kunci:")
print(ringkasan.to_string())


Statistik Deskriptif — Variabel Kunci:
                        Rata-rata     Median   Std Dev       Min        Max
Variabel                                                                   
usia                        48.73      49.00     17.90     18.00      79.00
pendapatan_bulanan     131,704.28 131,916.00 68,120.73 15,005.00 249,989.00
total_belanja          150,216.56 149,918.00 61,358.89  1,819.00 298,502.00
pesanan_online_bulanan      24.68      25.00     14.43      0.00      49.00
kunjungan_toko_bulanan       9.48       9.00      5.73      0.00      19.00
sensitivitas_diskon          5.50       6.00      2.88      1.00      10.00
skor_loyalitas_merek         5.53       6.00      2.85      1.00      10.00
jam_internet_harian          6.01       6.00      1.98      1.00      12.00


In [4]:
# Distribusi variabel kategorikal
print("=" * 50)
print("DISTRIBUSI JENIS KELAMIN")
print("=" * 50)
print(df["jenis_kelamin"].value_counts())

print("\n" + "=" * 50)
print("DISTRIBUSI TINGKAT KOTA")
print("=" * 50)
print(df["tingkat_kota"].value_counts())

print("\n" + "=" * 50)
print("DISTRIBUSI PREFERENSI BELANJA")
print("=" * 50)
print(df["preferensi_belanja"].value_counts())
print(df["preferensi_belanja"].value_counts(normalize=True).apply(lambda x: f"{x:.1%}"))


DISTRIBUSI JENIS KELAMIN
jenis_kelamin
Pria       3966
Wanita     3931
Lainnya    3892
Name: count, dtype: int64

DISTRIBUSI TINGKAT KOTA
tingkat_kota
Kota Tier 1    3982
Kota Tier 3    3949
Kota Tier 2    3858
Name: count, dtype: int64

DISTRIBUSI PREFERENSI BELANJA
preferensi_belanja
Toko      10244
Online     1176
Hybrid      369
Name: count, dtype: int64
preferensi_belanja
Toko      86.9%
Online    10.0%
Hybrid     3.1%
Name: proportion, dtype: object


---
## 3. Segmentasi Pelanggan (K-Means Clustering)

In [5]:
# Segmentasi pelanggan dengan K-Means
fitur_cluster = ["pendapatan_bulanan", "pesanan_online_bulanan",
                 "sensitivitas_diskon", "skor_loyalitas_merek"]

scaler  = StandardScaler()
X_s     = scaler.fit_transform(df[fitur_cluster])
km      = KMeans(n_clusters=4, random_state=42, n_init=10)
df["segmen_id"] = km.fit_predict(X_s)

# Beri nama segmen berdasarkan profil (urutkan berdasarkan total_belanja)
profil = df.groupby("segmen_id")[fitur_cluster + ["total_belanja"]].mean().sort_values("total_belanja", ascending=False)
print("Profil Segmen (rata-rata per kelompok):")
print(profil.to_string())

nama_segmen = {}
sorted_ids = profil.index.tolist()
labels_map = ["Pembelanja Premium", "Belanja Seimbang", "Pemburu Diskon", "Pengunjung Kasual"]
for i, sid in enumerate(sorted_ids):
    nama_segmen[sid] = labels_map[i]

df["segmen_pelanggan"] = df["segmen_id"].map(nama_segmen)
print("\nJumlah per segmen:")
print(df["segmen_pelanggan"].value_counts())


Profil Segmen (rata-rata per kelompok):
           pendapatan_bulanan  pesanan_online_bulanan  sensitivitas_diskon  skor_loyalitas_merek  total_belanja
segmen_id                                                                                                      
3                  132,591.71                   12.79                 5.45                  8.25     151,294.23
1                  129,443.37                   36.09                 8.25                  5.48     150,507.66
0                  132,238.87                   36.50                 2.77                  5.56     150,073.81
2                  132,657.92                   12.60                 5.41                  2.80     148,961.34

Jumlah per segmen:
segmen_pelanggan
Belanja Seimbang      3055
Pemburu Diskon        2932
Pembelanja Premium    2920
Pengunjung Kasual     2882
Name: count, dtype: int64


---
## 4. Visualisasi

### Visual 1 — Pola Belanja: Online vs. Toko per Kelompok Usia

**Insight:** Melihat bagaimana konsumen mendistribusikan anggaran belanja antara saluran online dan toko fisik.


In [6]:
# Melt data untuk violin plot
melt_df = df.melt(
    id_vars=["kelompok_usia"],
    value_vars=["rata_belanja_online", "rata_belanja_toko"],
    var_name="Saluran",
    value_name="Jumlah Belanja (Rp)"
)
melt_df["Saluran"] = melt_df["Saluran"].map({
    "rata_belanja_online": "Belanja Online",
    "rata_belanja_toko":   "Belanja Toko"
})

fig1 = px.violin(
    melt_df, x="kelompok_usia", y="Jumlah Belanja (Rp)", color="Saluran",
    box=True, points=False,
    color_discrete_map={"Belanja Online": "#00C9A7", "Belanja Toko": "#FF6B6B"},
    labels={"kelompok_usia": "Kelompok Usia"},
    title="<b>Distribusi Belanja Online vs Toko per Kelompok Usia</b>",
    template="plotly_dark",
    category_orders={"kelompok_usia": ["18-25","26-35","36-45","46-55","56-65","66-79"]},
)
fig1.update_traces(meanline_visible=True)
fig1.update_layout(
    paper_bgcolor="#0d1b2a", plot_bgcolor="#0d1b2a",
    font_family="Inter, sans-serif",
    height=420,
    legend=dict(bgcolor="rgba(0,0,0,0)"),
)
fig1.show()

# Insight
avg_by_age = df.groupby("kelompok_usia")[["rata_belanja_online","rata_belanja_toko"]].mean()
print("\nRata-rata belanja per kelompok usia:")
print(avg_by_age.apply(lambda x: x.apply(lambda v: f"Rp {v:,.0f}")))
print("\n💡 INSIGHT: Kelompok usia 26-35 memiliki rata-rata belanja online tertinggi.")
print("   Belanja toko lebih merata di semua kelompok usia, mencerminkan kebiasaan")
print("   berbelanja konvensional yang masih kuat di semua generasi.")



Rata-rata belanja per kelompok usia:
              rata_belanja_online rata_belanja_toko
kelompok_usia                                      
18-25                   Rp 73,333         Rp 75,808
26-35                   Rp 74,615         Rp 76,370
36-45                   Rp 73,392         Rp 74,916
46-55                   Rp 74,494         Rp 76,606
56-65                   Rp 75,031         Rp 75,376
66-79                   Rp 75,718         Rp 75,175

💡 INSIGHT: Kelompok usia 26-35 memiliki rata-rata belanja online tertinggi.
   Belanja toko lebih merata di semua kelompok usia, mencerminkan kebiasaan
   berbelanja konvensional yang masih kuat di semua generasi.


### Visual 2 — Preferensi Saluran Belanja per Tingkat Kota

**Insight:** Bagaimana karakteristik kota memengaruhi pilihan saluran belanja konsumen?


In [7]:
pref_kota = df.groupby(["tingkat_kota", "preferensi_belanja"]).size().reset_index(name="jumlah")

fig2 = px.sunburst(
    pref_kota, path=["tingkat_kota", "preferensi_belanja"], values="jumlah",
    color="preferensi_belanja",
    color_discrete_map={"Online": "#00C9A7", "Toko": "#FF6B6B", "Hybrid": "#FFD93D"},
    title="<b>Preferensi Saluran Belanja per Tingkat Kota</b>",
)
fig2.update_traces(textinfo="label+percent entry", insidetextfont_size=11)
fig2.update_layout(
    paper_bgcolor="#0d1b2a", plot_bgcolor="#0d1b2a",
    font_family="Inter, sans-serif",
    height=420,
)
fig2.show()

print("Distribusi preferensi belanja:")
cross = pd.crosstab(df["tingkat_kota"], df["preferensi_belanja"], normalize="index")
print((cross * 100).round(1).to_string())
print("\n💡 INSIGHT: Belanja Toko mendominasi di semua tingkat kota (>85%).")
print("   Konsumen Kota Tier 1 memiliki proporsi belanja Online tertinggi —")
print("   mencerminkan infrastruktur digital dan kepercayaan platform yang lebih baik.")


Distribusi preferensi belanja:
preferensi_belanja  Hybrid  Online  Toko
tingkat_kota                            
Kota Tier 1           3.50   10.00 86.50
Kota Tier 2           2.80    9.80 87.40
Kota Tier 3           3.10   10.10 86.80

💡 INSIGHT: Belanja Toko mendominasi di semua tingkat kota (>85%).
   Konsumen Kota Tier 1 memiliki proporsi belanja Online tertinggi —
   mencerminkan infrastruktur digital dan kepercayaan platform yang lebih baik.


### Visual 3 — Pendapatan vs. Total Belanja per Segmen Pelanggan

**Insight:** Apakah pendapatan menjadi driver utama pola belanja? Bagaimana segmen terbentuk?


In [8]:
sample_df = df.sample(2000, random_state=42)

fig3 = px.scatter(
    sample_df, x="pendapatan_bulanan", y="total_belanja",
    color="segmen_pelanggan",
    size="pesanan_online_bulanan",
    hover_data=["usia", "preferensi_belanja", "tingkat_kota"],
    color_discrete_sequence=["#0F4C81","#00C9A7","#FF6B6B","#FFD93D"],
    title="<b>Pendapatan Bulanan vs. Total Belanja — Segmentasi Pelanggan</b>",
    labels={
        "pendapatan_bulanan": "Pendapatan Bulanan (Rp)",
        "total_belanja": "Total Belanja (Rp)",
        "segmen_pelanggan": "Segmen",
    },
    opacity=0.65,
    trendline="lowess",
    template="plotly_dark",
)
fig3.update_layout(
    paper_bgcolor="#0d1b2a", plot_bgcolor="#0d1b2a",
    font_family="Inter, sans-serif",
    height=450,
    legend=dict(bgcolor="rgba(0,0,0,0)"),
)
fig3.update_xaxes(gridcolor="rgba(255,255,255,0.07)")
fig3.update_yaxes(gridcolor="rgba(255,255,255,0.07)")
fig3.show()

corr_val = df["pendapatan_bulanan"].corr(df["total_belanja"])
print(f"\nKorelasi Pearson (Pendapatan vs Total Belanja): {corr_val:.4f}")
print("\nRata-rata total belanja per segmen:")
print(df.groupby("segmen_pelanggan")["total_belanja"].agg(["mean","count"]).sort_values("mean", ascending=False).to_string())
print("\n💡 INSIGHT: Korelasi rendah antara pendapatan dan belanja menunjukkan bahwa")
print("   motivasi belanja sangat dipengaruhi oleh perilaku (diskon, loyalitas merek),")
print("   bukan hanya kemampuan finansial. Segmen 'Pembelanja Premium' konsisten di high-spend.")



Korelasi Pearson (Pendapatan vs Total Belanja): 0.0082

Rata-rata total belanja per segmen:
                         mean  count
segmen_pelanggan                    
Pembelanja Premium 151,294.23   2920
Belanja Seimbang   150,507.66   3055
Pemburu Diskon     150,073.81   2932
Pengunjung Kasual  148,961.34   2882

💡 INSIGHT: Korelasi rendah antara pendapatan dan belanja menunjukkan bahwa
   motivasi belanja sangat dipengaruhi oleh perilaku (diskon, loyalitas merek),
   bukan hanya kemampuan finansial. Segmen 'Pembelanja Premium' konsisten di high-spend.


### Visual 4 — Profil Segmen Pelanggan (K-Means, k=4)

**Insight:** Empat tipe konsumen dominan dan karakteristik unik masing-masing.


In [9]:
seg_agg = df.groupby("segmen_pelanggan").agg(
    jumlah=("usia", "count"),
    avg_belanja=("total_belanja", "mean"),
    avg_pesanan=("pesanan_online_bulanan", "mean"),
    avg_pendapatan=("pendapatan_bulanan", "mean"),
    avg_diskon=("sensitivitas_diskon", "mean"),
    avg_loyalitas=("skor_loyalitas_merek", "mean"),
).reset_index().sort_values("avg_belanja", ascending=True)

fig4 = go.Figure()
colors = ["#845EC2", "#0F4C81", "#00C9A7", "#FF6B6B"]
for i, row in seg_agg.iterrows():
    fig4.add_trace(go.Bar(
        y=[row["segmen_pelanggan"]],
        x=[row["avg_belanja"]],
        orientation="h",
        marker_color=colors[int(i) % 4],
        text=f"Rp {row['avg_belanja']/1000:.0f}K (n={row['jumlah']:,})",
        textposition="outside",
        name=row["segmen_pelanggan"],
    ))

fig4.update_layout(
    title="<b>Rata-rata Total Belanja per Segmen Pelanggan</b>",
    xaxis_title="Rata-rata Total Belanja (Rp)",
    paper_bgcolor="#0d1b2a", plot_bgcolor="#0d1b2a",
    font=dict(color="#E0E0E0", family="Inter, sans-serif"),
    height=350, showlegend=False,
    margin=dict(l=150, r=80, t=60, b=40),
    xaxis=dict(gridcolor="rgba(255,255,255,0.07)"),
)
fig4.show()

print("\nProfil lengkap setiap segmen:")
print(seg_agg.set_index("segmen_pelanggan")[[
    "jumlah","avg_pendapatan","avg_belanja","avg_pesanan","avg_diskon","avg_loyalitas"
]].rename(columns={
    "jumlah":"N", "avg_pendapatan":"Pendapatan", "avg_belanja":"Total Belanja",
    "avg_pesanan":"Pesanan Online", "avg_diskon":"Sens.Diskon", "avg_loyalitas":"Loyalitas"
}).to_string())
print("\n💡 INSIGHT: Masing-masing segmen memiliki profil unik. 'Pemburu Diskon' yang besar")
print("   secara jumlah bisa menjadi profit center jika dikelola dengan program loyalitas tepat.")



Profil lengkap setiap segmen:
                       N  Pendapatan  Total Belanja  Pesanan Online  Sens.Diskon  Loyalitas
segmen_pelanggan                                                                           
Pengunjung Kasual   2882  132,657.92     148,961.34           12.60         5.41       2.80
Pemburu Diskon      2932  132,238.87     150,073.81           36.50         2.77       5.56
Belanja Seimbang    3055  129,443.37     150,507.66           36.09         8.25       5.48
Pembelanja Premium  2920  132,591.71     151,294.23           12.79         5.45       8.25

💡 INSIGHT: Masing-masing segmen memiliki profil unik. 'Pemburu Diskon' yang besar
   secara jumlah bisa menjadi profit center jika dikelola dengan program loyalitas tepat.


### Visual 5 — Radar: Profil Perilaku per Tingkat Kota

**Insight:** Perbandingan multi-dimensi karakteristik konsumen berdasarkan ukuran kota.


In [10]:
radar_vars = {
    "Jam Internet": "jam_internet_harian",
    "Sens. Diskon": "sensitivitas_diskon",
    "Loyalitas Merek": "skor_loyalitas_merek",
    "Kepercayaan Digital": "kepercayaan_pembayaran_online",
    "Melek Teknologi": "skor_melek_teknologi",
    "Pembelian Impulsif": "skor_pembelian_impulsif",
}

kota_grp = df.groupby("tingkat_kota")[[v for v in radar_vars.values()]].mean()
kota_grp.columns = list(radar_vars.keys())
cats = list(radar_vars.keys())

fig5 = go.Figure()
colors_r = ["#00C9A7", "#FF6B6B", "#FFD93D"]
for i, (kota, row) in enumerate(kota_grp.iterrows()):
    vals = list(row) + [row.iloc[0]]
    fig5.add_trace(go.Scatterpolar(
        r=vals, theta=cats + [cats[0]],
        name=kota,
        line_color=colors_r[i],
        fill="toself",
        opacity=0.75,
    ))

fig5.update_layout(
    polar=dict(
        radialaxis=dict(visible=True, gridcolor="rgba(255,255,255,0.1)", color="rgba(255,255,255,0.5)"),
        angularaxis=dict(gridcolor="rgba(255,255,255,0.1)"),
        bgcolor="rgba(0,0,0,0)",
    ),
    title="<b>Profil Perilaku Konsumen per Tingkat Kota</b>",
    paper_bgcolor="#0d1b2a",
    font=dict(color="#E0E0E0", family="Inter, sans-serif"),
    legend=dict(bgcolor="rgba(0,0,0,0)"),
    height=460,
)
fig5.show()

print("\nNilai rata-rata per kota:")
print(kota_grp.to_string())
print("\n💡 INSIGHT: Kota Tier 1 memimpin di Melek Teknologi dan Kepercayaan Digital.")
print("   Kota Tier 3 memiliki Sensitivitas Diskon tertinggi — konsumen lebih price-driven.")
print("   Profil ini penting untuk strategi diferensiasi kampanye pemasaran per wilayah.")



Nilai rata-rata per kota:
              Jam Internet  Sens. Diskon  Loyalitas Merek  Kepercayaan Digital  Melek Teknologi  Pembelian Impulsif
tingkat_kota                                                                                                       
Kota Tier 1           6.00          5.46             5.46                 5.47             5.48                5.41
Kota Tier 2           6.03          5.52             5.58                 5.48             5.59                5.58
Kota Tier 3           6.01          5.52             5.55                 5.55             5.53                5.47

💡 INSIGHT: Kota Tier 1 memimpin di Melek Teknologi dan Kepercayaan Digital.
   Kota Tier 3 memiliki Sensitivitas Diskon tertinggi — konsumen lebih price-driven.
   Profil ini penting untuk strategi diferensiasi kampanye pemasaran per wilayah.


### Visual 6 — Heatmap Korelasi Antar Variabel Perilaku

**Insight:** Hubungan tersembunyi antar variabel perilaku belanja konsumen.


In [11]:
corr_cols = {
    "Pendapatan": "pendapatan_bulanan",
    "Total Belanja": "total_belanja",
    "Pesanan Online": "pesanan_online_bulanan",
    "Kunjungan Toko": "kunjungan_toko_bulanan",
    "Sens. Diskon": "sensitivitas_diskon",
    "Loyalitas Merek": "skor_loyalitas_merek",
    "Melek Teknologi": "skor_melek_teknologi",
    "Impulse Buying": "skor_pembelian_impulsif",
    "Kepercayaan Digital": "kepercayaan_pembayaran_online",
}

corr_df  = df[[v for v in corr_cols.values()]].rename(columns={v: k for k, v in corr_cols.items()})
corr_mat = corr_df.corr()

fig6 = go.Figure(data=go.Heatmap(
    z=corr_mat.values,
    x=list(corr_cols.keys()),
    y=list(corr_cols.keys()),
    colorscale="RdBu_r", zmid=0,
    text=np.round(corr_mat.values, 2),
    texttemplate="%{text}",
    textfont_size=9,
    colorbar=dict(len=0.8, tickfont=dict(color="#ccc", size=9)),
))
fig6.update_layout(
    title="<b>Matriks Korelasi — Variabel Perilaku Belanja</b>",
    paper_bgcolor="#0d1b2a", plot_bgcolor="#0d1b2a",
    font=dict(color="#E0E0E0", family="Inter, sans-serif"),
    height=500, margin=dict(l=100, r=20, t=60, b=120),
    xaxis=dict(tickangle=-35),
)
fig6.show()

print("\nKorelasi terkuat (|r| > 0.05):")
corr_pairs = corr_mat.unstack().reset_index()
corr_pairs.columns = ["Var1", "Var2", "Korelasi"]
strong = corr_pairs[
    (corr_pairs["Var1"] != corr_pairs["Var2"]) &
    (corr_pairs["Korelasi"].abs() > 0.05)
].drop_duplicates(subset=["Korelasi"]).sort_values("Korelasi", ascending=False).head(10)
print(strong.to_string(index=False))
print("\n💡 INSIGHT: Melek Teknologi berkorelasi positif dengan Kepercayaan Digital.")
print("   Ini berarti program edukasi teknologi akan meningkatkan adopsi pembayaran digital.")
print("   Loyalitas Merek berkorelasi negatif dengan Impulse Buying — pelanggan loyal lebih selektif.")



Korelasi terkuat (|r| > 0.05):
Empty DataFrame
Columns: [Var1, Var2, Korelasi]
Index: []

💡 INSIGHT: Melek Teknologi berkorelasi positif dengan Kepercayaan Digital.
   Ini berarti program edukasi teknologi akan meningkatkan adopsi pembayaran digital.
   Loyalitas Merek berkorelasi negatif dengan Impulse Buying — pelanggan loyal lebih selektif.


---
## 5. Ringkasan Insight & Rekomendasi Strategis

| No | Area | Insight | Rekomendasi Aksi |
|----|------|---------|-----------------|
| 1 | **Saluran Belanja** | 86% konsumen masih memilih toko fisik | Perkuat pengalaman in-store, tambahkan layanan klik-dan-ambil |
| 2 | **Segmen Digital** | Konsumen online = pesanan lebih banyak | Optimalkan UX app, program loyalitas digital |
| 3 | **Premium Segment** | Pembelanja Premium: belanja tertinggi | Program VIP, personalisasi produk, free shipping |
| 4 | **Pemburu Diskon** | Segmen terbesar, sens. diskon tinggi | Flash sale, bundling, gamification di aplikasi |
| 5 | **Kota Tier 1** | Melek teknologi & kepercayaan digital tinggi | Prioritaskan ekspansi e-commerce di kota besar |
| 6 | **Kota Tier 3** | Sensitivitas harga sangat tinggi | Strategi penetrasi harga, program cashback lokal |
| 7 | **Kelompok Usia 26-45** | Segmen paling produktif secara belanja | Personalisasi konten, konten UGC, referral program |

### 🎯 Prioritas Utama (Dampak Tertinggi)

1. **Digitalisasi konsumen toko** → Hybrid shopping experience (BOPIS, QR payment)  
2. **Segmentasi kampanye** → Berbeda per profil pelanggan (Premium vs Kasual)  
3. **Diferensiasi per wilayah** → Tier 1 digital-first, Tier 3 price-driven

> *Data ini memberi kita peta jalan yang jelas untuk mengoptimalkan strategi retail di era hybrid shopping.*


In [12]:
# Ringkasan numerik akhir
print("=" * 60)
print("RINGKASAN EKSEKUTIF — TREN BELANJA KONSUMEN 2026")
print("=" * 60)
print(f"Total konsumen dianalisis   : {len(df):,}")
print(f"Rata-rata usia              : {df['usia'].mean():.1f} tahun")
print(f"Rata-rata pendapatan        : Rp {df['pendapatan_bulanan'].mean():,.0f}/bulan")
print(f"Rata-rata total belanja     : Rp {df['total_belanja'].mean():,.0f}/bulan")
print(f"Preferensi Toko             : {(df['preferensi_belanja']=='Toko').mean()*100:.1f}%")
print(f"Preferensi Online           : {(df['preferensi_belanja']=='Online').mean()*100:.1f}%")
print(f"Preferensi Hybrid           : {(df['preferensi_belanja']=='Hybrid').mean()*100:.1f}%")
print(f"Rata-rata pesanan online    : {df['pesanan_online_bulanan'].mean():.1f}x/bulan")
print(f"Rata-rata kunjungan toko    : {df['kunjungan_toko_bulanan'].mean():.1f}x/bulan")
print()
print("Jumlah per segmen pelanggan:")
for seg, n in df["segmen_pelanggan"].value_counts().items():
    pct = n / len(df) * 100
    print(f"  {seg:<25} : {n:,} ({pct:.1f}%)")
print()
print("✅ Analisis selesai! Jalankan dashboard.py untuk visualisasi interaktif.")
print("   Perintah: python dashboard.py")


RINGKASAN EKSEKUTIF — TREN BELANJA KONSUMEN 2026
Total konsumen dianalisis   : 11,789
Rata-rata usia              : 48.7 tahun
Rata-rata pendapatan        : Rp 131,704/bulan
Rata-rata total belanja     : Rp 150,217/bulan
Preferensi Toko             : 86.9%
Preferensi Online           : 10.0%
Preferensi Hybrid           : 3.1%
Rata-rata pesanan online    : 24.7x/bulan
Rata-rata kunjungan toko    : 9.5x/bulan

Jumlah per segmen pelanggan:
  Belanja Seimbang          : 3,055 (25.9%)
  Pemburu Diskon            : 2,932 (24.9%)
  Pembelanja Premium        : 2,920 (24.8%)
  Pengunjung Kasual         : 2,882 (24.4%)

✅ Analisis selesai! Jalankan dashboard.py untuk visualisasi interaktif.
   Perintah: python dashboard.py
